# NexusGS 3-View Sparse Reconstruction (Mitsubishi Data)

NexusGS의 **Epipolar Depth Prior** 기반 sparse view synthesis를 Mitsubishi 데이터에 적용합니다.

### 파이프라인 구조 (README 기반)

NexusGS 공식 파이프라인:
```
1. LLFF 데이터 (images/ + sparse/0/ + poses_bounds.npy)
2. colmap_llff.py → n_views 선택 + COLMAP triangulation + dense fusion
3. FlowFormer++ → optical flow (.flo) 생성
4. train.py → flow 기반 depth 추정 + PCD 초기화 + 학습
```

우리 적용:
```
1. Mitsubishi calib_*.ini → COLMAP sparse/0/ (cameras.txt, images.txt)
2. rgb_*.png → images/ (배경 제거) → images_2/ (2x 다운샘플)
3. GT depth → optical flow 합성 (FlowFormer++ 대체)
4. train.py 실행 (run_llff.sh 동일 파라미터)
```

### 데이터 구조 (NexusGS 기대 형식)
```
scene/
├── sparse/0/           ← cameras.txt, images.txt, points3D.txt
├── images/             ← 원본 해상도 (800x800, 배경 제거)
├── images_2/           ← 2x 다운샘플 (400x400)
└── 3_views/
    └── flow/           ← optical flow (.flo, Middlebury format)
```

---
## 0. 환경 설정

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
import os

WORK_DIR = "/content/NexusGS"
DATA_DIR = "/content/data"
OUTPUT_DIR = "/content/output"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# NexusGS 클론
if not os.path.exists(WORK_DIR):
    !git clone https://github.com/USMizuki/NexusGS {WORK_DIR}

os.chdir(WORK_DIR)

# 의존성 설치
!pip install -q plyfile tqdm tensorboard gdown open3d imageio matplotlib

# NexusGS 핵심: confidence-weighted rasterizer (CUDA 빌드)
print("=" * 50)
print("Building diff-gaussian-rasterization-confidence...")
print("=" * 50)
!pip install submodules/diff-gaussian-rasterization-confidence

# simple-knn: NexusGS 내장 서브모듈은 Colab에서 빌드 실패
# camenduru 포크 사용 (TriaGS 파이프라인과 동일)
print("=" * 50)
print("Building simple-knn (camenduru fork)...")
print("=" * 50)
!pip install git+https://github.com/camenduru/simple-knn

# ================================================================
# 호환성 패치: distCUDA2 반환 형식 차이
#
#   NexusGS 내장 simple-knn:  distCUDA2() → tuple(distances, indices)
#   camenduru simple-knn:     distCUDA2() → distances (Tensor only)
#
# NexusGS 코드가 result[0]으로 인덱싱 →
#   tuple이면: distances 텐서 (N,) ← 정상
#   Tensor이면: 첫 번째 원소 scalar ← _scaling이 [1,3]이 되어 densify 에러
# ================================================================
gauss_model_path = os.path.join(WORK_DIR, "scene", "gaussian_model.py")
with open(gauss_model_path, 'r') as f:
    lines = f.readlines()

patched = False
with open(gauss_model_path, 'w') as f:
    for line in lines:
        if 'distCUDA2(fused_point_cloud)[0]' in line:
            indent = line[:len(line) - len(line.lstrip())]
            f.write(f"{indent}dist2_raw = distCUDA2(fused_point_cloud)\n")
            f.write(f"{indent}dist2 = torch.clamp_min(dist2_raw[0] if isinstance(dist2_raw, tuple) else dist2_raw, 0.0000001)\n")
            patched = True
        else:
            f.write(line)

if patched:
    print("[PATCH] gaussian_model.py: distCUDA2 호환성 패치 적용 완료")
else:
    print("[PATCH] 패치 대상 라인 없음 (이미 적용됨)")

# 패치 확인: 해당 라인 출력
!grep -n "dist2" {WORK_DIR}/scene/gaussian_model.py | head -5

# 빌드 검증
try:
    from diff_gaussian_rasterization import GaussianRasterizer
    print("[OK] diff-gaussian-rasterization-confidence")
except ImportError as e:
    print(f"[FAIL] diff-gaussian-rasterization: {e}")

try:
    from simple_knn._C import distCUDA2
    print("[OK] simple-knn")
except ImportError as e:
    print(f"[FAIL] simple-knn: {e}")

# ======================================================
# [PATCH 2] train.py: 배경 마스킹 패치
# 문제: 전경 11%, 배경 89% → Loss가 배경에 지배됨
# 해결: white_background 모드에서 gt_image가 흰색인 영역을 bg_mask로 설정,
#       L1 loss와 SSIM에 mask를 전달하여 배경을 Loss에서 제외
# ======================================================
train_py = os.path.join(WORK_DIR, "train.py")
with open(train_py, 'r') as f:
    train_content = f.read()

# 이미 패치되었는지 확인
if '# [BG_MASK_PATCH]' not in train_content:
    # 1) llff 모드에서도 bg_mask 생성 (white_background일 때)
    old_loss = """        render_pkg = render(viewpoint_cam, gaussians, pipe, background)
        image, viewspace_point_tensor, visibility_filter, radii = render_pkg["render"], render_pkg["viewspace_points"], render_pkg["visibility_filter"], render_pkg["radii"]

        # Loss
        Ll1 =  l1_loss_mask(image, gt_image)
        loss = ((1.0 - opt.lambda_dssim) * Ll1 + opt.lambda_dssim * (1.0 - ssim(image, gt_image)))"""

    new_loss = """        render_pkg = render(viewpoint_cam, gaussians, pipe, background)
        image, viewspace_point_tensor, visibility_filter, radii = render_pkg["render"], render_pkg["viewspace_points"], render_pkg["visibility_filter"], render_pkg["radii"]

        # [BG_MASK_PATCH] 배경 마스킹: white_background에서 흰색 영역 제외
        if args.dataset_type == 'llff' and dataset.white_background:
            bg_mask = (gt_image.min(0, keepdim=True).values > 0.99)
            fg_mask = (~bg_mask).float()
        else:
            fg_mask = None

        # Loss (fg_mask 적용: 배경 제외)
        Ll1 =  l1_loss_mask(image, gt_image, fg_mask)
        loss = ((1.0 - opt.lambda_dssim) * Ll1 + opt.lambda_dssim * (1.0 - ssim(image, gt_image, fg_mask)))"""

    if old_loss in train_content:
        train_content = train_content.replace(old_loss, new_loss)
        with open(train_py, 'w') as f:
            f.write(train_content)
        print("[PATCH 2] train.py: 배경 마스킹 패치 적용 완료")
        print("  - llff + white_background일 때 gt>0.99 → bg_mask")
        print("  - l1_loss_mask(image, gt, fg_mask)")
        print("  - ssim(image, gt, fg_mask)")
    else:
        print("[PATCH 2] 패치 대상 코드 없음 (구조 변경됨?)")
else:
    print("[PATCH 2] 이미 적용됨, 건너뜀")

print("\n환경 설정 완료")

---
## 1. 데이터 다운로드

TriaGS 파이프라인과 동일한 Mitsubishi 데이터를 사용합니다.

In [ ]:
import gdown

# RGB + Camera calibration
RGB_ZIP = f"{DATA_DIR}/dataset.zip"
if not os.path.exists(RGB_ZIP):
    gdown.download("https://drive.google.com/uc?id=1dnj1s-mqIuS6OcdSr5CczBr9u8yBzUUB", RGB_ZIP, quiet=False)

# GT Depth (flow 합성에 사용)
DEPTH_ZIP = f"{DATA_DIR}/depth.zip"
if not os.path.exists(DEPTH_ZIP):
    gdown.download("https://drive.google.com/uc?id=1KmXCzBYv_mkPmZnWka1ThCZSNHTyivKQ", DEPTH_ZIP, quiet=False)

!unzip -q -o {RGB_ZIP} -d {DATA_DIR}/raw_rgb
!unzip -q -o {DEPTH_ZIP} -d {DATA_DIR}/raw_depth
print("다운로드 완료")

In [ ]:
import glob

def find_data_root(base_dir, prefix):
    for root, dirs, files in os.walk(base_dir):
        if any(f.startswith(prefix) for f in files):
            return root
    return None

RGB_DATA_DIR = find_data_root(f"{DATA_DIR}/raw_rgb", "rgb_")
DEPTH_DATA_DIR = find_data_root(f"{DATA_DIR}/raw_depth", "depth_raw_")
print(f"RGB: {RGB_DATA_DIR} ({len(glob.glob(os.path.join(RGB_DATA_DIR, 'rgb_*.png')))} files)")
print(f"Depth: {DEPTH_DATA_DIR} ({len(glob.glob(os.path.join(DEPTH_DATA_DIR, 'depth_raw_*.png')))} files)")

---
## 2. COLMAP 구조 생성

NexusGS는 `sparse/0/`에서 COLMAP 형식(binary 우선, text fallback)을 읽습니다.  
우리 `calib_*.ini`를 COLMAP text 형식으로 변환합니다.

또한 `images/` (원본) + `images_2/` (2x 다운샘플)을 준비합니다.  
NexusGS 기본값은 `images_8`이지만, 800x800 원본에서 8x는 100x100으로 너무 작습니다.

In [ ]:
#@title 설정 파라미터 {display-mode: "form"}

N_VIEWS = 3          #@param {type:"integer"}
NUM_TOTAL = 200      #@param {type:"integer"}
DEPTH_SCALE = 0.01   #@param {type:"number"}
LLFFHOLD = 8         #@param {type:"integer"}
#@markdown ↑ NexusGS LLFF test split 간격 (매 N번째 = test)

SCENE_SCALE = 40.0   #@param {type:"number"}
#@markdown ↑ Scene normalization factor
#@markdown - 원본: 카메라 반경 400, cameras_extent ≈ 453 (LLFF 대비 45x)
#@markdown - /40 적용: 카메라 반경 10, cameras_extent ≈ 11.3 (LLFF 수준)
#@markdown - NexusGS의 position_lr, densification 등 모든 하이퍼파라미터가 LLFF 기준이므로 스케일 맞춤 필수
#@markdown - flow 파일은 변경 불필요 (픽셀 좌표는 스케일 무관)

SCENE_DIR = f"{DATA_DIR}/nexus_scene"
MODEL_PATH = f"{OUTPUT_DIR}/nexus_{N_VIEWS}views"

print(f"Scene: {SCENE_DIR}")
print(f"Output: {MODEL_PATH}")
print(f"설정: {N_VIEWS}-view sparse, {NUM_TOTAL}장, llffhold={LLFFHOLD}")
print(f"Scene scale: 1/{SCENE_SCALE:.0f} (cameras_extent ≈ {453/SCENE_SCALE:.1f})")

In [ ]:
import numpy as np
import cv2
from pathlib import Path
import shutil

# === 유틸리티 함수 ===

def parse_calib(filepath):
    """Mitsubishi calib_*.ini 파싱 → K, R, T"""
    config = {}
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('[') or line.startswith('#'):
                continue
            key, value = line.split('=', 1)
            config[key] = value
    k_vals = list(map(float, config['k_matrix'].split(':')[-1].split(',')))
    K = np.array(k_vals).reshape(3, 3)
    r_vals = list(map(float, config['r_matrix'].split(':')[-1].split(',')))
    R = np.array(r_vals).reshape(3, 3)
    t_vals = list(map(float, config['t_vector'].split(':')[-1].split(',')))
    T = np.array(t_vals)
    return K, R, T, int(config['width']), int(config['height'])

def rotmat2qvec(R):
    """Rotation matrix → COLMAP quaternion (w, x, y, z)"""
    Rxx, Ryx, Rzx, Rxy, Ryy, Rzy, Rxz, Ryz, Rzz = R.flat
    K_mat = np.array([
        [Rxx - Ryy - Rzz, 0, 0, 0],
        [Ryx + Rxy, Ryy - Rxx - Rzz, 0, 0],
        [Rzx + Rxz, Rzy + Ryz, Rzz - Rxx - Ryy, 0],
        [Ryz - Rzy, Rzx - Rxz, Rxy - Ryx, Rxx + Ryy + Rzz]]) / 3.0
    eigvals, eigvecs = np.linalg.eigh(K_mat)
    qvec = eigvecs[[3, 0, 1, 2], np.argmax(eigvals)]
    if qvec[0] < 0:
        qvec *= -1
    return qvec

def write_flo(filename, flow):
    """Middlebury .flo format 저장 (NexusGS readFlow와 호환)"""
    h, w, _ = flow.shape
    with open(filename, 'wb') as f:
        np.array([202021.25], np.float32).tofile(f)  # magic number
        np.array([w], np.int32).tofile(f)
        np.array([h], np.int32).tofile(f)
        flow.astype(np.float32).tofile(f)

def get_index(filepath):
    return Path(filepath).stem.split('_')[-1]

print("유틸리티 로드 완료")

In [ ]:
# === 파일 매칭 + 이미지 선택 ===

calib_dict = {get_index(f): f for f in sorted(glob.glob(f"{RGB_DATA_DIR}/calib_*.ini"))}
rgb_dict   = {get_index(f): f for f in sorted(glob.glob(f"{RGB_DATA_DIR}/rgb_*.png"))}
depth_dict = {get_index(f): f for f in sorted(glob.glob(f"{DEPTH_DATA_DIR}/depth_raw_*.png"))}

all_indices = sorted(set(calib_dict.keys()) & set(rgb_dict.keys()) & set(depth_dict.keys()))
N_ALL = len(all_indices)
print(f"RGB+Depth+Calib 매칭: {N_ALL}개")

# 균일 샘플링
sample_idx = np.linspace(0, N_ALL - 1, NUM_TOTAL, dtype=int)
selected_indices = [all_indices[i] for i in sample_idx]
print(f"선택: {len(selected_indices)}개")

In [ ]:
# === COLMAP 구조 생성 + 배경 제거 + 다운샘플 ===

if os.path.exists(SCENE_DIR):
    shutil.rmtree(SCENE_DIR)

images_dir    = os.path.join(SCENE_DIR, "images")      # 원본 800x800
images_2_dir  = os.path.join(SCENE_DIR, "images_2")     # 2x 다운샘플 400x400
sparse_dir    = os.path.join(SCENE_DIR, "sparse", "0")  # NexusGS: sparse/0/
os.makedirs(images_dir, exist_ok=True)
os.makedirs(images_2_dir, exist_ok=True)
os.makedirs(sparse_dir, exist_ok=True)

# Intrinsics (모든 카메라 동일)
K0, _, _, W0, H0 = parse_calib(calib_dict[selected_indices[0]])
fx, fy, cx, cy = K0[0, 0], K0[1, 1], K0[0, 2], K0[1, 2]

# --- cameras.txt (intrinsics는 스케일 무관) ---
with open(os.path.join(sparse_dir, "cameras.txt"), 'w') as f:
    f.write("# Camera list with one line of data per camera:\n")
    f.write("# CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]\n")
    f.write(f"1 PINHOLE {W0} {H0} {fx} {fy} {cx} {cy}\n")

# --- images.txt + 이미지 처리 ---
calib_cache = {}  # flow 합성용 (원본 R, T 저장)

with open(os.path.join(sparse_dir, "images.txt"), 'w') as f:
    f.write("# Image list with two lines of data per image:\n")
    f.write("# IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME\n")
    f.write("# POINTS2D[] as (X, Y, POINT3D_ID)\n")

    for img_id, idx in enumerate(selected_indices, start=1):
        K, R, T, W, H = parse_calib(calib_dict[idx])
        qvec = rotmat2qvec(R)
        img_name = f"rgb_{idx}.png"

        # 원본 R, T 저장 (flow 합성은 원본 좌표계에서 수행)
        calib_cache[idx] = {'K': K, 'R': R, 'T': T, 'W': W, 'H': H}

        # 배경 제거 + 흰색 배경: depth > 0이면 전경, 아니면 흰색
        # 실험 분석 결과: 전경 ~11%, 배경 ~89%
        # 검정 배경 + bg_color=[0,0,0]이면 Loss의 89%가 배경 매칭에 소모됨
        # 흰색 배경 + --white_background로 렌더러 bg와 GT를 일치시킴
        rgb_img = cv2.imread(rgb_dict[idx])
        depth_raw = cv2.imread(depth_dict[idx], cv2.IMREAD_UNCHANGED)
        fg_mask = (depth_raw > 0).astype(np.float32)
        if fg_mask.ndim == 2:
            fg_mask = fg_mask[..., np.newaxis]
        # 전경: 원본 RGB, 배경: 흰색(255)
        rgb_white_bg = (rgb_img * fg_mask + 255.0 * (1.0 - fg_mask)).astype(np.uint8)

        # images/ (원본 해상도)
        cv2.imwrite(os.path.join(images_dir, img_name), rgb_white_bg)

        # images_2/ (2x 다운샘플)
        h2, w2 = H // 2, W // 2
        rgb_half = cv2.resize(rgb_white_bg, (w2, h2), interpolation=cv2.INTER_AREA)
        cv2.imwrite(os.path.join(images_2_dir, img_name), rgb_half)

        # ★ Scene normalization: T를 SCENE_SCALE로 나눠서 기록
        # flow 파일은 원본 좌표계로 생성 → 픽셀 대응은 동일
        # NexusGS가 축소된 T로 삼각측량 → depth, PCD 모두 1/SCENE_SCALE 스케일
        T_scaled = T / SCENE_SCALE
        f.write(f"{img_id} {qvec[0]} {qvec[1]} {qvec[2]} {qvec[3]} "
                f"{T_scaled[0]} {T_scaled[1]} {T_scaled[2]} 1 {img_name}\n")
        f.write("\n")

# --- points3D.txt (빈 파일) ---
with open(os.path.join(sparse_dir, "points3D.txt"), 'w') as f:
    f.write("# 3D point list\n")

print(f"[OK] COLMAP 구조 생성 (SCENE_SCALE={SCENE_SCALE})")
print(f"  sparse/0/: cameras.txt, images.txt (T/{SCENE_SCALE:.0f}), points3D.txt")
print(f"  images/:   {len(os.listdir(images_dir))}장 ({W0}x{H0}, 배경 제거)")
print(f"  images_2/: {len(os.listdir(images_2_dir))}장 ({W0//2}x{H0//2})")
print(f"  예상 cameras_extent: ~{453/SCENE_SCALE:.1f} (LLFF 수준)")

---
## 3. Optical Flow 생성

NexusGS는 학습 뷰 쌍의 optical flow (`.flo`)를 입력으로 받습니다.  
정석은 [FlowFormer++](https://github.com/XiaoyuShi97/FlowFormerPlusPlus)이지만,  
우리 데이터에는 GT depth가 있으므로 **정확한 flow를 직접 합성**합니다.

```
FlowFormer++ 경로:  Image pair → DNN → estimated flow (.flo)
우리 경로:          Image pair + GT depth → geometric projection → exact flow (.flo)
```

### 합성 원리
Source 뷰의 픽셀 (u,v)에 대해:
1. GT depth `d`로 3D 역투영: `X_cam = K^{-1} [u,v,1] * d`
2. World 좌표 변환: `X_world = R_s^T (X_cam - T_s)`
3. Target 뷰로 투영: `X_cam_t = R_t X_world + T_t` → `(u', v')`
4. Flow = `(u'-u, v'-v)`

### Flow 파일 네이밍 (NexusGS readColmapCameras 기준)
```
{n}_views/flow/{rgb_name}_{source_train_idx}_{target_train_idx}.flo
```

In [ ]:
# === NexusGS 내부 뷰 선택 로직 재현 ===
#
# NexusGS readColmapSceneInfo (dataset_type='llff', eval=True, n_views=3):
#   1) 모든 이미지를 이름순 정렬
#   2) idx % llffhold == 0 → test,  나머지 → train pool
#   3) train pool에서 np.linspace(0, len-1, n_views) → 학습 뷰 선택
#
# flow 파일은 이 3개 학습 뷰에 대해서만 필요

all_image_names = sorted(os.listdir(images_dir))
n_total = len(all_image_names)

# Step 1: LLFF split
train_pool_pos = [i for i in range(n_total) if i % LLFFHOLD != 0]
test_pool_pos  = [i for i in range(n_total) if i % LLFFHOLD == 0]

# Step 2: n_views 선택
idx_sub = [round(x) for x in np.linspace(0, len(train_pool_pos) - 1, N_VIEWS)]
nexus_train_global_pos = [train_pool_pos[i] for i in idx_sub]
nexus_train_names = [all_image_names[p] for p in nexus_train_global_pos]
nexus_train_idx = [Path(name).stem.split('_')[-1] for name in nexus_train_names]

print(f"=== NexusGS 뷰 선택 시뮬레이션 ===")
print(f"전체 {n_total}장 → test {len(test_pool_pos)}장 / train pool {len(train_pool_pos)}장")
print(f"\n학습 뷰 {N_VIEWS}장 (flow 생성 대상):")
for k, (gpos, name) in enumerate(zip(nexus_train_global_pos, nexus_train_names)):
    print(f"  train[{k}]: {name}  (전체 위치 {gpos}/{n_total})")

In [ ]:
# === GT Depth 기반 Optical Flow 합성 ===

flow_dir = os.path.join(SCENE_DIR, f"{N_VIEWS}_views", "flow")
os.makedirs(flow_dir, exist_ok=True)

# 학습 뷰 데이터 로드
train_views = []
for idx in nexus_train_idx:
    c = calib_cache[idx]
    depth_raw = cv2.imread(depth_dict[idx], cv2.IMREAD_UNCHANGED)
    depth_m = depth_raw.astype(np.float32) * DEPTH_SCALE
    train_views.append({
        'K': c['K'], 'R': c['R'], 'T': c['T'],
        'depth': depth_m, 'idx': idx,
        'W': c['W'], 'H': c['H']
    })

print(f"Flow 생성: {N_VIEWS}개 뷰 x {N_VIEWS-1}개 타겟 = {N_VIEWS*(N_VIEWS-1)}개 파일\n")

for s_k, src in enumerate(train_views):
    K_s, R_s, T_s = src['K'], src['R'], src['T']
    depth_s = src['depth']
    H, W = src['H'], src['W']
    rgb_name = f"rgb_{src['idx']}"

    for t_k, tgt in enumerate(train_views):
        if s_k == t_k:
            continue

        K_t, R_t, T_t = tgt['K'], tgt['R'], tgt['T']

        # 픽셀 그리드
        u_grid, v_grid = np.meshgrid(
            np.arange(W, dtype=np.float32),
            np.arange(H, dtype=np.float32)
        )

        # Source 픽셀 → Camera 좌표
        z = depth_s
        x_cam = (u_grid - K_s[0, 2]) / K_s[0, 0] * z
        y_cam = (v_grid - K_s[1, 2]) / K_s[1, 1] * z
        pts_cam = np.stack([x_cam.ravel(), y_cam.ravel(), z.ravel()], axis=0)

        # Camera → World → Target Camera
        pts_world = R_s.T @ (pts_cam - T_s[:, None])
        pts_cam_t = R_t @ pts_world + T_t[:, None]

        # Target 픽셀 좌표
        z_t = pts_cam_t[2]
        valid = z_t > 1e-6

        u_t = np.zeros_like(z_t)
        v_t = np.zeros_like(z_t)
        u_t[valid] = K_t[0, 0] * pts_cam_t[0, valid] / z_t[valid] + K_t[0, 2]
        v_t[valid] = K_t[1, 1] * pts_cam_t[1, valid] / z_t[valid] + K_t[1, 2]

        # Flow = target - source
        flow_u = (u_t - u_grid.ravel()).reshape(H, W)
        flow_v = (v_t - v_grid.ravel()).reshape(H, W)

        # 배경(depth=0) + 카메라 뒤쪽 → 0으로 마스킹
        invalid = (~valid.reshape(H, W)) | (depth_s <= 0)
        flow_u[invalid] = 0.0
        flow_v[invalid] = 0.0

        flow = np.stack([flow_u, flow_v], axis=-1).astype(np.float32)

        # .flo 저장
        flo_name = f"{rgb_name}_{s_k}_{t_k}.flo"
        write_flo(os.path.join(flow_dir, flo_name), flow)

        n_valid = int((~invalid).sum())
        print(f"  {flo_name}: valid {n_valid}/{H*W} "
              f"u[{flow_u[~invalid].min():.1f},{flow_u[~invalid].max():.1f}] "
              f"v[{flow_v[~invalid].min():.1f},{flow_v[~invalid].max():.1f}]")

print(f"\n[OK] {flow_dir} ({len(os.listdir(flow_dir))} files)")

In [ ]:
# === Flow 시각화 (sanity check) ===
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(N_VIEWS, N_VIEWS, figsize=(4*N_VIEWS, 4*N_VIEWS))

for s_k in range(N_VIEWS):
    for t_k in range(N_VIEWS):
        ax = axes[s_k][t_k]
        if s_k == t_k:
            # 대각선: 원본 이미지
            img = Image.open(os.path.join(images_dir, nexus_train_names[s_k]))
            ax.imshow(img)
            ax.set_title(f"View {s_k}", fontsize=9)
        else:
            # 비대각선: flow magnitude
            flo_path = os.path.join(flow_dir, f"rgb_{nexus_train_idx[s_k]}_{s_k}_{t_k}.flo")
            with open(flo_path, 'rb') as f:
                np.fromfile(f, np.float32, 1)  # magic
                w = np.fromfile(f, np.int32, 1)[0]
                h = np.fromfile(f, np.int32, 1)[0]
                flo = np.fromfile(f, np.float32).reshape(h, w, 2)
            magnitude = np.sqrt(flo[:,:,0]**2 + flo[:,:,1]**2)
            magnitude[magnitude < 1e-3] = np.nan  # 배경 숨김
            ax.imshow(magnitude, cmap='hot')
            ax.set_title(f"Flow {s_k}→{t_k}", fontsize=9)
        ax.axis('off')

plt.suptitle("Training Views & Synthesized Flow Magnitudes", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# === 최종 데이터 구조 검증 ===
print("=== 데이터 구조 ===")
for root, dirs, files in os.walk(SCENE_DIR):
    level = root.replace(SCENE_DIR, '').count(os.sep)
    indent = '  ' * level
    subdir = os.path.basename(root)
    n_files = len(files)
    if n_files > 5:
        print(f"{indent}{subdir}/ ({n_files} files)")
    else:
        print(f"{indent}{subdir}/")
        for f in sorted(files):
            print(f"{indent}  {f}")

---
## 4. NexusGS 학습

`scripts/run_llff.sh`와 동일한 파라미터를 사용합니다.

NexusGS 학습 흐름:
1. `readColmapSceneInfo` → COLMAP 카메라 로드 + train/test split + flow 로드
2. `compute_depth_by_flow` → flow에서 epipolar triangulation으로 depth 추정
3. `construct_pcd` → flow depth에서 초기 3D 포인트 클라우드 생성
4. Training loop → RGB L1+SSIM loss, split_num=4 densification

In [ ]:
#@title 학습 파라미터 (run_llff.sh 기준) {display-mode: "form"}

ITERATIONS = 30000         #@param {type:"integer"}
SPLIT_NUM = 4              #@param {type:"integer"}
VALID_DIS_THRESHOLD = 1.0  #@param {type:"number"}
DROP_RATE = 1.0            #@param {type:"number"}
NEAR_N = 2                 #@param {type:"integer"}
IMAGES_DIR_NAME = "images_2"  #@param ["images", "images_2"]
#@markdown - `images`: 원본 800x800
#@markdown - `images_2`: 2x 다운샘플 400x400 (권장)

os.chdir(WORK_DIR)

# run_llff.sh와 동일한 구조
cmd = f"""python train.py \\
    --source_path {SCENE_DIR} \\
    --model_path {MODEL_PATH} \\
    --eval \\
    --n_views {N_VIEWS} \\
    --iterations {ITERATIONS} \\
    --densify_until_iter {ITERATIONS} \\
    --position_lr_max_steps {ITERATIONS} \\
    --dataset_type llff \\
    --images {IMAGES_DIR_NAME} \\
    --split_num {SPLIT_NUM} \\
    --valid_dis_threshold {VALID_DIS_THRESHOLD} \\
    --drop_rate {DROP_RATE} \\
    --near_n {NEAR_N} \\
    --save_iterations {ITERATIONS} \\
    --test_iterations 5000 10000 {ITERATIONS} \
    --white_background"""

print(cmd)

In [ ]:
os.chdir(WORK_DIR)
!{cmd}

---
## 5. 렌더링 + 메트릭

NexusGS `render.py`는 `cfg_args`에서 학습 설정을 자동 로드합니다.

In [ ]:
os.chdir(WORK_DIR)

# test 뷰 렌더링 + depth 저장
!python render.py \
    --model_path {MODEL_PATH} \
    --iteration {ITERATIONS} \
    --skip_train \
    --render_depth \
    --white_background

In [ ]:
os.chdir(WORK_DIR)

# PSNR / SSIM / LPIPS
!python metrics.py -m {MODEL_PATH}

In [ ]:
# 결과 요약
import json

results_file = os.path.join(MODEL_PATH, "results.json")
if os.path.exists(results_file):
    with open(results_file) as f:
        results = json.load(f)
    print("=" * 50)
    print(f"  NexusGS {N_VIEWS}-view | {IMAGES_DIR_NAME} | {ITERATIONS} iters")
    print("=" * 50)
    for scene, methods in results.items():
        for method, metrics in methods.items():
            print(f"  [{method}]")
            for metric, value in metrics.items():
                print(f"    {metric}: {value:.6f}")
else:
    print("results.json 없음 — metrics.py 실행 확인")

---
## 6. 시각화

In [ ]:
# === 학습 입력 뷰 (3장) ===
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, N_VIEWS, figsize=(5 * N_VIEWS, 5))
for k, name in enumerate(nexus_train_names):
    img = Image.open(os.path.join(images_dir, name))
    axes[k].imshow(img)
    axes[k].set_title(f"Train [{k}]: {name}", fontsize=10)
    axes[k].axis('off')
plt.suptitle(f"NexusGS {N_VIEWS}-View Input", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# === GT vs Render 비교 ===
render_dir = os.path.join(MODEL_PATH, "test", f"ours_{ITERATIONS}", "renders")
gt_dir     = os.path.join(MODEL_PATH, "test", f"ours_{ITERATIONS}", "gt")

if not os.path.exists(render_dir):
    print(f"렌더링 결과 없음: {render_dir}")
else:
    gt_files = sorted([f for f in os.listdir(gt_dir) if f.endswith('.png')])
    n_show = min(6, len(gt_files))

    fig, axes = plt.subplots(n_show, 3, figsize=(15, 4 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    for i, fname in enumerate(gt_files[:n_show]):
        gt_img = Image.open(os.path.join(gt_dir, fname))
        render_img = Image.open(os.path.join(render_dir, fname))

        axes[i, 0].imshow(gt_img)
        axes[i, 0].set_title(f"GT: {fname}", fontsize=9)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(render_img)
        axes[i, 1].set_title("NexusGS", fontsize=9)
        axes[i, 1].axis('off')

        depth_name = fname.replace('.png', '_depth.png')
        depth_path = os.path.join(render_dir, depth_name)
        if os.path.exists(depth_path):
            depth_img = Image.open(depth_path)
            axes[i, 2].imshow(depth_img)
            axes[i, 2].set_title("Depth", fontsize=9)
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_PATH, "nexus_comparison.png"), dpi=150, bbox_inches='tight')
    plt.show()

---
## 7. GT Depth 기반 기하학적 평가

NexusGS 렌더링 depth vs GT depth를 비교합니다.  
(TriaGS 파이프라인과 동일한 평가 방식)

In [ ]:
render_dir = os.path.join(MODEL_PATH, "test", f"ours_{ITERATIONS}", "renders")
depth_npy_files = sorted(glob.glob(os.path.join(render_dir, "*_depth.npy")))

if len(depth_npy_files) == 0:
    print("depth .npy 없음 — --render_depth 확인")
else:
    test_names = [all_image_names[p] for p in test_pool_pos]
    test_idx_map = {Path(n).stem: Path(n).stem.split('_')[-1] for n in test_names}

    abs_rel_list, rmse_list = [], []

    for npy_path in depth_npy_files:
        view_name = os.path.basename(npy_path).replace('_depth.npy', '')
        if view_name not in test_idx_map:
            continue
        orig_idx = test_idx_map[view_name]
        if orig_idx not in depth_dict:
            continue

        gt_depth = cv2.imread(depth_dict[orig_idx], cv2.IMREAD_UNCHANGED).astype(np.float32) * DEPTH_SCALE
        rd_depth = np.load(npy_path).squeeze()

        if gt_depth.shape != rd_depth.shape:
            gt_depth = cv2.resize(gt_depth, (rd_depth.shape[1], rd_depth.shape[0]),
                                  interpolation=cv2.INTER_NEAREST)

        valid = (gt_depth > 0) & (rd_depth > 0)
        if valid.sum() < 100:
            continue

        gt_v, rd_v = gt_depth[valid], rd_depth[valid]
        scale = np.median(gt_v) / np.median(rd_v)
        rd_aligned = rd_v * scale

        abs_rel_list.append(np.mean(np.abs(rd_aligned - gt_v) / gt_v))
        rmse_list.append(np.sqrt(np.mean((rd_aligned - gt_v) ** 2)))

    if abs_rel_list:
        print(f"=== Depth 평가 ({len(abs_rel_list)} test views) ===")
        print(f"  Abs Rel: {np.mean(abs_rel_list):.4f} +/- {np.std(abs_rel_list):.4f}")
        print(f"  RMSE:    {np.mean(rmse_list):.4f} +/- {np.std(rmse_list):.4f}")
    else:
        print("매칭 test 뷰 없음")